# Modeling: Accepted vs Rejected PR Trace Classification

This notebook trains a random baseline, logistic regression baseline, and an XGBoost baseline on the flat feature tables exported by `python -m pipeline features`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

In [ ]:
base = Path('data/processed')
train_df = pd.read_parquet(base / 'dataset_mvp_v0.1.train.features.parquet')
val_df = pd.read_parquet(base / 'dataset_mvp_v0.1.val.features.parquet')
test_df = pd.read_parquet(base / 'dataset_mvp_v0.1.test.features.parquet')

target = 'accepted'
drop_cols = ['example_id', 'repo', 'pr_number', target]
feature_cols = [column for column in train_df.columns if column not in drop_cols]

X_train = train_df[feature_cols]
y_train = train_df[target]
X_val = val_df[feature_cols]
y_val = val_df[target]
X_test = test_df[feature_cols]
y_test = test_df[target]

In [ ]:
categorical_features = ['top_language', 'author_association']
numeric_features = [column for column in feature_cols if column not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_eval, y_eval):
    model.fit(X_train, y_train)
    pred = model.predict(X_eval)
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_eval)[:, 1]
    else:
        proba = pred
    print(f'## {name}')
    print('F1:', f1_score(y_eval, pred))
    print('ROC-AUC:', roc_auc_score(y_eval, proba))
    print('Confusion matrix:\n', confusion_matrix(y_eval, pred))
    print(classification_report(y_eval, pred))
    return model

In [ ]:
random_baseline = DummyClassifier(strategy='stratified', random_state=42)
evaluate_model('Random baseline', random_baseline, X_train, y_train, X_test, y_test)

logreg = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
evaluate_model('Logistic regression', logreg, X_train, y_train, X_test, y_test)

In [ ]:
X_train_xgb = pd.get_dummies(X_train, columns=categorical_features, dummy_na=True)
X_val_xgb = pd.get_dummies(X_val, columns=categorical_features, dummy_na=True)
X_test_xgb = pd.get_dummies(X_test, columns=categorical_features, dummy_na=True)

X_val_xgb = X_val_xgb.reindex(columns=X_train_xgb.columns, fill_value=0)
X_test_xgb = X_test_xgb.reindex(columns=X_train_xgb.columns, fill_value=0)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
)
xgb_model = evaluate_model('XGBoost', xgb_model, X_train_xgb, y_train, X_test_xgb, y_test)

In [ ]:
feature_importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_train_xgb.columns,
).sort_values(ascending=False)
feature_importance.head(20)